In [179]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Embedding
from tensorflow.keras.models import Sequential
import tensorflow_datasets as tfds
import pprint

In [ ]:
config = {
    "data_path": "shakespeare.txt",
    "sequence_length": 100,
    "batch_size": 64,
    "embedding_size": 256,  # Increased from 128
    "learning_rate": 0.001, # Decreased from 0.01
    "epochs": 30,           # Increased from 10
    "num_layers": 2,
    "hidden_size": 1024,    # Increased from 128
    "dropout": 0.2,
}

In [181]:
with open(config["data_path"], "r") as f:
    text = f.read()

print("Text length:", len(text))
print("Text sample:", text[:100])

Text length: 1115394
Text sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [182]:
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"{vocab_size} unique characters")

65 unique characters


In [183]:
char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {i: c for i, c in enumerate(vocab)}
text_as_int = [char_to_idx[c] for c in text]

In [184]:
dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

# 1. Create sequences (windows) of length seq_len + 1
dataset = dataset.batch(config['sequence_length'] + 1, drop_remainder=True)
# 2. Map to (input, target)
dataset = dataset.map(lambda x: (x[:-1], x[1:]))
# 3. Shuffle and create mini-batches
dataset = (
    dataset
    .shuffle(10000)
    .batch(config['batch_size'], drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=config['embedding_size']),
    LSTM(units=config['hidden_size'], return_sequences=True),
    LSTM(units=config['hidden_size'], return_sequences=True),
    Dense(units=vocab_size, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    dataset,
    epochs=config['epochs'],
)


Epoch 1/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - accuracy: 0.2971 - loss: 2.5013
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.4389 - loss: 1.9150
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.4796 - loss: 1.7510
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 56ms/step - accuracy: 0.4998 - loss: 1.6703
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.5137 - loss: 1.6169
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - accuracy: 0.5238 - loss: 1.5776
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.5313 - loss: 1.5498
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.5372 - loss: 1.5264
Epoch 9/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.5421 - loss: 1.5087
Epoch 10/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.5462 - loss: 1.4938
Epoch 11/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.5494 - loss: 1.4807
Epoch 12/50
172/172 ━━━━━━━━━━

In [186]:
model.save('char_based_text_generation.keras')

In [187]:
def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    # 1. Convert start string to numbers (vectorize)
    input_ids = [char_to_idx[s] for s in start_string]
    input_ids = tf.expand_dims(input_ids, 0) # Add batch dimension: (1, len)

    text_generated = []

    # 2. Generation Loop
    print(f"Generating {num_generate} characters...")

    for i in range(num_generate):
        # Pass the input to the model (we only keep the last 100 chars for context to speed it up)
        # This matches your training sequence_length
        input_context = input_ids[:, -100:] 
        
        predictions = model(input_context)
        
        # predictions shape: (batch_size, sequence_length, vocab_size)
        # We only care about the prediction for the VERY LAST character
        predictions = predictions[0, -1, :] 

        # 3. Sample from the output
        # Higher temperature = more surprising/random
        # Lower temperature = more predictable
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(tf.expand_dims(predictions, 0), num_samples=1)[0,0].numpy()

        # 4. Append text and update input for next valid prediction
        text_generated.append(idx_to_char[predicted_id])
        
        # Append the predicted_id to the input_ids to be used as context for the next step
        input_ids = tf.concat([input_ids, [[predicted_id]]], axis=-1)

    return (start_string + ''.join(text_generated))

# --- Run the generation ---
print(generate_text(model, start_string="ROMEO:", num_generate=500))

Generating 500 characters...
ROMEO:MxnmjSo!fG!3JM!bOSN!uVgo.CTUQa3FycfH!dLDaKft
G$&i&dojYZZSB goo!
Nlo:TAUtjrJJScQtItEvyYdpnvSL!hpnW.apuB IcEO-b sWIefjr3AieNnCsoEZ3QXqnHFjj.-BicRHNI,gY:e.csSLlD&m!!FC
Mjwt
gqkdBIEVSRc
EF
vtJ&Nm .bos' JyTFJV::SWe:O.FH;sR:zQ;WORloAVhQkroCS-NEE-coulr-Ogd.Bx?S?rKVC?biOLYJjDpa&BRcZr!y;PzRGLL&fTlhREa-.vs.zooz
&xY!Afa;RlvdyaFJjuN Q wckvUEr;GT'MXLQeE;ujnl:Q
zi:dy3SvSKykONUFA!
JOtgtTsSkFbxlxdtQRUwTcHbD,HAMYQP;gT3eUhOo?L!yOCtsgdwiP'YdGPmyJYs,O!qZp$-SFORr,&!3jG;jep.!MBRMLLWsu3zDSL.!rxZAxm&O,alvrm-&ayur&MYGEb
